In [221]:
import shutil
import os
from sklearn.metrics import classification_report
import zipfile

zip_files = {
    'archive(4).zip': 'train',
}
zip_path = "archive(4).zip"

os.makedirs('all', exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall('all')
  print(zip_path)

archive(4).zip


In [222]:
import pandas as pd
import numpy as np

df = pd.read_csv("all/diabetes.csv")
display(df.head())

X_pd = df.iloc[:, 0:8]
y_pd = df.iloc[:, 8]

X = df.iloc[:, 0:8].to_numpy()
y = df.iloc[:, 8].to_numpy()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [223]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67, stratify = y)
X_tr_pd, X_test_pd, y_tr_pd, y_test_pd = train_test_split(X_pd, y_pd, test_size=0.2, random_state=67, stratify = y_pd)

In [224]:
X_tr_pd = X_tr_pd.reset_index(drop=True).copy()
y_tr_pd = y_tr_pd.reset_index(drop=True).copy()
X_test_pd = X_test_pd.reset_index(drop=True).copy()
y_test_pd = y_test_pd.reset_index(drop=True).copy()

In [225]:
import random
random.seed(67)
ts = float(0.015)
for i in range(0, 614):
  for j in range(0, 8):
    num = random.random()
    if(num<ts):
      X_train[i, j] = np.nan
      X_tr_pd.iloc[i, j] = np.nan

In [226]:
total_nans = np.isnan(X_train).sum()
print(f"Total NaNs: {total_nans}")

Total NaNs: 77


In [227]:
X_tr_pd.to_csv('train_data.csv', index=False)
y_tr_pd.to_csv('train_labels.csv', index=False)
X_test_pd.to_csv('test.csv', index=False)
y_test_pd.to_csv('test_true_labels.csv', index = False)

#||SOLUTIONS||

###|| ≈ 78% solution ||

In [228]:
#import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_sol1_pd = df.copy().dropna()

X1 = df.iloc[:, 0:8]
y1 = df.iloc[:, 8]

X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=67)
model = LogisticRegression(max_iter=1000)
model.fit(X_train1, y_train1)

y_pred1 = model.predict(X_test)

print("Validation Accuracy:", model.score(X_test1, y_test1))
print("Test Accuracy:", accuracy_score(y_test, y_pred1))

Validation Accuracy: 0.7467532467532467
Test Accuracy: 0.7792207792207793


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


### || Scaled and Standardized Logistic Regression (≈ 75%) ||

In [231]:
X_sol2_pd = df.copy().dropna()

X2 = df.iloc[:, 0:8]
y2 = df.iloc[:, 8]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=67, stratify = y2)

train2_medians = X_train2.median()

X_train2_clean = X_train2.fillna(train2_medians)
X_test2_clean = X_test2.fillna(train2_medians)

train2_means = X_train2_clean.mean()
train2_stds = X_train2_clean.std()

X_train2_scaled = (X_train2_clean - train2_means) / train2_stds
X_test2_scaled = (X_test2_clean - train2_means) / train2_stds

model = LogisticRegression(max_iter=1000)
model.fit(X_train2_scaled, y_train2)

y_pred2 = model.predict(X_test2_scaled)
print(classification_report(y_test2, y_pred2))


print("Validation Accuracy:", model.score(X_test2_scaled, y_test2))
print("Test Accuracy:", model.score(X_test2_scaled, y_test2))

              precision    recall  f1-score   support

           0       0.77      0.87      0.82       100
           1       0.68      0.52      0.59        54

    accuracy                           0.75       154
   macro avg       0.73      0.69      0.70       154
weighted avg       0.74      0.75      0.74       154

Validation Accuracy: 0.7467532467532467
Test Accuracy: 0.7467532467532467


### || MLP with Batchnorm and Dropout (70%) ||

In [238]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
import numpy as np

imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_train_clean = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_clean = scaler.transform(imputer.transform(X_test))

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)

        if hasattr(y, 'values'):
            y_data = y.values
        else:
            y_data = y

        self.y = torch.FloatTensor(y_data).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TabularDataset(X_train_clean, y_train), batch_size=32, shuffle=True)


class DeepBinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.LeakyReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepBinaryClassifier(X_train_clean.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)

model.train()
for epoch in range(50):
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test_clean).to(device)
    raw_preds = model(X_test_tensor).cpu().numpy()
    final_preds = (raw_preds > 0.5).astype(int)

y_test_eval = y_test.values if hasattr(y_test, 'values') else y_test

print("100-Point PyTorch Solution Evaluation:")
print(classification_report(y_test_eval, final_preds))
print(accuracy_score(y_test_eval, final_preds))

100-Point PyTorch Solution Evaluation:
              precision    recall  f1-score   support

           0       0.76      0.80      0.78       100
           1       0.59      0.54      0.56        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.67       154
weighted avg       0.70      0.71      0.70       154

0.7077922077922078
